# Restaurant Annual Turnover Prediction — October 12 Improvement

**Student register number:** submission

**Student name:** submission

**Team name:** submission

Predict annual turnover in INR for the 500 test restaurants. The submission has
exactly `Registration Number` and `Annual Turnover` as its two columns.

**Colab instructions:** upload this notebook, use a CPU runtime, and Run all.
Upload the four supplied CSV inputs when prompted. The final cell downloads the
prediction CSV. Save the executed `.ipynb` through Colab's File menu.
Validation fits many small models and can take several minutes on a Colab CPU.

## 1. Objective, actual leaderboard result, and model selection

This is **regression**. The metric is `sqrt(mean((actual - prediction)**2))`
on the original turnover values in INR. Lower is better. Every validation
restaurant, including extreme turnover values, is included.

The October 12 screenshot shows **one submission**, RMSE **12,204,615.6438253**,
and **rank 7**. The same submission was ranked 3 on October 11. Its score did not
change; competing entries moved ahead. The leader shown scores **11,946,659.0702364**.
The earlier improved model (v2) and this new candidate have no portal result in the
provided screenshots. A local score cannot establish their hidden-test ranking.

The earlier round compared 38 model settings. This round tested 42 further settings:
14 CatBoost configurations averaged over three seeds, 20 spline/linear/kernel
settings, and 8 Random Forest settings. A coarse blend grid on seed-42 development
predictions selected **50% of the v2 ensemble + 50% of a three-seed subsampled
CatBoost ensemble**. Random Forest worsened development RMSE; smoother models
provided only a small gain and were not retained.

All settings below are frozen before the confirmation checks. Seeds 71, 2026 and
303 were already used in earlier work. Seed 909's split was reserved during this
round's selection. All splits reuse the same labeled restaurants, whose labels
also informed development; these are robustness checks, **not an untouched external
test or a nested estimate of the entire model-selection process**.

Huber losses and a training-only cap are regularization choices. Validation targets
are never capped, transformed, removed, or replaced. No hidden test targets are used.

In [ ]:
import os, sys, json, hashlib, importlib.util, subprocess, warnings, copy
from pathlib import Path
packages={'numpy':'numpy==2.5.4','pandas':'pandas==3.0.6','sklearn':'scikit-learn==1.9.1',
          'catboost':'catboost==1.2.10','scipy':'scipy','matplotlib':'matplotlib'}
missing=[v for k,v in packages.items() if importlib.util.find_spec(k) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, catboost, scipy
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
warnings.filterwarnings('ignore',category=FutureWarning)
FILE_STEM='submission'
OUTPUT_DIR=Path.cwd()
print('Python:',sys.version.split()[0])
print('Versions:',{k:v.__version__ for k,v in {'numpy':np,'pandas':pd,'sklearn':sklearn,'catboost':catboost,'scipy':scipy}.items()})
print('Tested versions: numpy 2.5.4, pandas 3.0.6, scikit-learn 1.9.1, catboost 1.2.10.')

## 2. Read the source files and enforce their schema

The dictionary identifies `Registration Number` as the **restaurant ID**. Student
details appear in the notebook and filenames, not in the 500 prediction rows.
The endorsement-column spelling difference in train and test is normalized.

In [ ]:
required=['Train_dataset.csv','Test_dataset.csv','Data_Dictionary.csv','Sample_Solution.csv']
DATA_DIR=Path.cwd()
if not all((DATA_DIR/f).exists() for f in required):
    local=Path('~/Downloads')
    if all((local/f).exists() for f in required): DATA_DIR=local
    else:
        try: from google.colab import files
        except ImportError: raise FileNotFoundError('Place the four source CSVs beside the notebook or set DATA_DIR.')
        files.upload(); DATA_DIR=Path.cwd()
assert all((DATA_DIR/f).exists() for f in required)
train=pd.read_csv(DATA_DIR/required[0]).rename(columns={'Endorsed By':'Endoresed By'})
test=pd.read_csv(DATA_DIR/required[1]); dictionary=pd.read_csv(DATA_DIR/required[2]); sample=pd.read_csv(DATA_DIR/required[3])
y=train['Annual Turnover'].to_numpy(float)
assert np.isfinite(y).all() and (y>=0).all()
assert 'Annual Turnover' not in test
assert train.drop(columns='Annual Turnover').columns.tolist()==test.columns.tolist()
assert len(test)==500 and test['Registration Number'].is_unique and train['Registration Number'].is_unique
assert set(train['Registration Number']).isdisjoint(test['Registration Number'])
assert sample.columns.tolist()==['Registration Number','Annual Turnover']
assert sample['Registration Number'].equals(test['Registration Number'])
input_hashes={f:hashlib.sha256((DATA_DIR/f).read_bytes()).hexdigest() for f in required}
print('Train:',train.shape,'Test:',test.shape)
print('PASS: schema, target values, restaurant IDs, sample headers and row order.')
print('Source SHA256 hashes:',json.dumps(input_hashes,indent=2))

## 3. Data findings and their implications

A small upper tail has a large effect on squared error. It motivates robust training
experiments but does not justify deleting difficult validation rows. Service and
comfort ratings also differ between train and test; a feature-removal variant was
evaluated in v2. These are possible reasons local RMSE differs from portal RMSE;
the cause cannot be verified without the hidden targets.

In [ ]:
print(pd.Series(y).describe(percentiles=[.5,.9,.95,.99]).round(0).to_string())
sq=(y-y.mean())**2
tail=y>100e6
print(f'Above INR 100m: {tail.sum()} restaurants; {100*sq[tail].sum()/sq.sum():.1f}% of mean-baseline squared error.')
print(pd.DataFrame({'Train':train[['Service','Comfortablility']].mean(),'Test':test[['Service','Comfortablility']].mean()}).round(3).to_string())
fig,ax=plt.subplots(figsize=(8,3.5))
ax.hist(y/1e6,bins=45,color='#2563eb',edgecolor='white')
ax.set(xlabel='Annual turnover (INR millions)',ylabel='Restaurants (log scale)',yscale='log',title='All training targets are retained for evaluation')
fig.tight_layout();plt.show()

## 4. Reproducible feature and model functions

Dates, cuisines, event availability, and missingness are handled deterministically.
The target and restaurant ID are excluded. Imputation, encoding and scaling are
fitted only on each training fold. CatBoost uses categorical values directly.

The new component uses Huber delta INR 40m, depth 5, 850 trees, learning rate 0.04,
L2 regularization 10, and Bernoulli row subsampling 0.7. Three seeds reduce variation
between training runs. The earlier ten-model mixture remains half of the final
prediction. This adds diversity without choosing predictions using hidden labels.

In [ ]:
TARGET = 'Annual Turnover'

ID = 'Registration Number'

EVENTS = ['Live Music Rating','Comedy Gigs Rating','Value Deals Rating','Live Sports Rating']

SOCIAL = ['Facebook Popularity Quotient','Instagram Popularity Quotient']

CORE = SOCIAL + ['Hygiene Rating','Restaurant Zomato Rating','Live Music Rating','Value for Money','Overall Restaurant Rating','Restaurant City Tier','Cuisine','Restaurant Theme','Restaurant Location','Endoresed By','City','Opening Year']

def features(raw, variant='base'):
    data=raw.drop(columns=[TARGET,ID],errors='ignore').copy()
    dates=pd.to_datetime(data.pop('Opening Day of Restaurant'),format='%d-%m-%Y',errors='coerce')
    data['Opening Year']=dates.dt.year.astype(float)
    data['Opening Month']=dates.dt.month.astype(float)
    for c in EVENTS:
        data['Offers '+c.replace(' Rating','')]=data[c].notna().astype(int)
    parts=data['Cuisine'].astype('string').fillna('Missing').str.split(',',n=1,expand=True)
    data['Cuisine 1']=parts[0].str.strip().str.lower()
    data['Cuisine 2']=parts.reindex(columns=[0,1])[1].astype('string').str.strip().str.lower()
    for c in ['Cuisine','City','Restaurant Location','Endoresed By','Restaurant Type','Restaurant Theme']:
        data[c]=data[c].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True).fillna('Missing')
    data['City Is Unknown']=(data['City']=='-1').astype(int)
    for c in data.select_dtypes(include=['object','string']).columns:
        data[c]=data[c].fillna('Missing').astype(str)
    if variant in ['engineered','engineered_drop']:
        data['City']=data['City'].replace({'bengaluru':'bangalore','gurugram':'gurgaon','bombay':'mumbai','calcutta':'kolkata','delhi':'new delhi'})
        data['Cuisine unordered']=['|'.join(sorted(v.split(','))) for v in data['Cuisine']]
        data['Opening Day']=dates.dt.day.astype(float)
        data['Opening Days Since 1970']=(dates-pd.Timestamp('1970-01-01')).dt.days.astype(float)
        data['Social mean']=data[SOCIAL].mean(axis=1)
        data['Social difference']=data[SOCIAL[0]]-data[SOCIAL[1]]
        data['Social missing count']=data[SOCIAL].isna().sum(axis=1)
        data['Rating missing count']=data.isna().sum(axis=1)
        data['Event count']=data[EVENTS].notna().sum(axis=1)
        data['Event mean']=data[EVENTS].mean(axis=1)
        data['Quality mean']=data[['Food Rating','Hygiene Rating','Overall Restaurant Rating']].mean(axis=1)
        data['Social x Zomato']=data['Social mean']*data['Restaurant Zomato Rating']
        data['Social x Hygiene']=data['Social mean']*data['Hygiene Rating']
        data['Quality x Value']=data['Quality mean']*data['Value for Money']
    if variant in ['drop_shift','engineered_drop']:
        data=data.drop(columns=['Service','Comfortablility'])
    if variant=='core': data=data[CORE]
    if variant=='ratings':
        keep=SOCIAL+['Resturant Tier','Restaurant Zomato Rating','Restaurant City Tier','Staff Responsivness','Value for Money','Hygiene Rating','Overall Restaurant Rating']+EVENTS+['Lively','Ambience','Privacy']
        data=data[keep]
    if variant=='no_city_date':
        data=data.drop(columns=['City','City Is Unknown','Opening Year','Opening Month'])
    return data

def rmse(y,p): return float(np.sqrt(np.mean(np.square(np.asarray(y)-np.asarray(p)))))

def sklearn_model(X,spec,seed):
    cats=X.select_dtypes(include=['object','string']).columns.tolist()
    nums=[c for c in X if c not in cats]
    num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler())
    if spec['kind']=='spline':
        num=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),SplineTransformer(n_knots=4,degree=2,include_bias=False),StandardScaler())
    pre=ColumnTransformer([('num',num,nums),('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=10,sparse_output=False),cats)])
    kind=spec['kind']
    if kind in ['ridge','spline']: model=Ridge(alpha=spec.get('alpha',100))
    elif kind=='svr': model=SVR(C=spec.get('C',20),epsilon=.5,kernel='rbf')
    elif kind=='hist': model=HistGradientBoostingRegressor(max_iter=300,max_leaf_nodes=10,max_depth=4,learning_rate=.035,l2_regularization=30,min_samples_leaf=60,random_state=seed)
    elif kind=='extra': model=ExtraTreesRegressor(n_estimators=400,min_samples_leaf=5,max_features=.7,n_jobs=4,random_state=seed)
    return make_pipeline(pre,model)

def fit_predict(X,y,train_idx,valid_idx,spec,seed,fixed_iterations=None,return_fitted=False):
    ys=np.asarray(y)[train_idx].copy()
    if 'cap' in spec: ys=np.minimum(ys,spec['cap'])
    if spec['kind']=='cat':
        params=dict(loss_function=spec.get('loss','RMSE'),eval_metric='RMSE',iterations=fixed_iterations or spec.get('iterations',1600),learning_rate=.04,depth=spec.get('depth',6),l2_leaf_reg=spec.get('l2',5),random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False)
        if 'boosting_type' in spec: params['boosting_type']=spec['boosting_type']
        if spec.get('monotone'): params['monotone_constraints']={**{c:1 for c in SOCIAL+['Hygiene Rating','Restaurant Zomato Rating']},'Resturant Tier':-1}
        model=CatBoostRegressor(**params)
        cat=X.select_dtypes(include=['object','string']).columns.tolist()
        stopping={} if fixed_iterations or 'iterations' in spec else dict(eval_set=(X.iloc[valid_idx],np.asarray(y)[valid_idx]),early_stopping_rounds=120)
        model.fit(X.iloc[train_idx],ys,cat_features=cat,**stopping)
        p=model.predict(X.iloc[valid_idx]); trees=model.tree_count_
    elif spec['kind'] in ['lgb','xgb']:
        pre=sklearn_model(X,dict(kind='ridge'),seed)[:-1]
        ztr=pre.fit_transform(X.iloc[train_idx]); zva=pre.transform(X.iloc[valid_idx])
        early=not (fixed_iterations or 'iterations' in spec)
        iterations=fixed_iterations or spec.get('iterations',1400)
        if spec['kind']=='lgb':
            from lightgbm import LGBMRegressor,early_stopping
            model=LGBMRegressor(n_estimators=iterations,learning_rate=.025,num_leaves=spec.get('leaves',15),min_child_samples=spec.get('leaf_samples',60),reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,subsample_freq=1,random_state=seed,n_jobs=4,verbosity=-1,objective=spec.get('objective','regression'))
            params=dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],eval_metric='rmse',callbacks=[early_stopping(100,verbose=False,first_metric_only=True)]) if early else {}
            model.fit(ztr,ys/1e6,**params)
            trees=model.best_iteration_ if early else iterations
        else:
            from xgboost import XGBRegressor
            params=dict(n_estimators=iterations,max_depth=spec.get('depth',3),learning_rate=.025,min_child_weight=30,reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,random_state=seed,n_jobs=4,tree_method='hist',objective=spec.get('objective','reg:squarederror'),eval_metric='rmse',base_score=float(ys.mean()/1e6))
            if early: params['early_stopping_rounds']=100
            if spec.get('objective')=='reg:pseudohubererror': params['huber_slope']=12
            model=XGBRegressor(**params)
            model.fit(ztr,ys/1e6,**(dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],verbose=False) if early else {}))
            trees=model.best_iteration+1 if early else iterations
        p=model.predict(zva)*1e6
    else:
        model=sklearn_model(X,spec,seed)
        model.fit(X.iloc[train_idx],ys/1e6)
        p=model.predict(X.iloc[valid_idx])*1e6; trees=0
    if return_fitted:
        fitted=dict(model=model,preprocessor=pre if spec['kind'] in ['lgb','xgb'] else None,scale=1 if spec['kind']=='cat' else 1e6)
        return np.maximum(p,0),trees,fitted
    return np.maximum(p,0),trees

def predict_fitted(fitted,X):
    transformed=fitted['preprocessor'].transform(X) if fitted['preprocessor'] is not None else X
    return np.maximum(fitted['model'].predict(transformed)*fitted['scale'],0)

def cat_fit(X, y, spec, seed):
    params=dict(iterations=spec['iterations'], learning_rate=.04, depth=spec['depth'],
                l2_leaf_reg=spec.get('l2',5), random_seed=seed, thread_count=2,
                verbose=False, allow_writing_files=False,
                loss_function=f"Huber:delta={spec['delta']*1e6:g}" if 'delta' in spec else 'RMSE')
    if spec.get('monotone'):
        params['monotone_constraints']={**{c:1 for c in SOCIAL+['Hygiene Rating','Restaurant Zomato Rating']},'Resturant Tier':-1}
    if spec.get('ordered'): params['boosting_type']='Ordered'
    if 'subsample' in spec: params.update(bootstrap_type='Bernoulli',subsample=spec['subsample'])
    if 'random_strength' in spec: params['random_strength']=spec['random_strength']
    fitted=CatBoostRegressor(**params)
    target=np.minimum(y,spec['cap']*1e6) if 'cap' in spec else y
    fitted.fit(X,target,cat_features=X.select_dtypes(include=['object','string']).columns.tolist())
    return fitted

In [ ]:
CONTRACT=json.loads('{"v2_configs": {"engineered": {"kind": "cat", "variant": "engineered", "depth": 6, "iterations": 260}, "engineered_drop": {"kind": "cat", "variant": "engineered_drop", "depth": 6, "iterations": 216}, "ordered_depth5": {"kind": "cat", "variant": "base", "depth": 5, "boosting_type": "Ordered", "iterations": 219}, "huber20_engineered": {"kind": "cat", "variant": "engineered_drop", "depth": 5, "loss": "Huber:delta=20000000", "iterations": 699}, "huber40": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 664}, "clip120": {"kind": "cat", "variant": "base", "depth": 5, "cap": 120000000.0, "iterations": 591}, "spline_ridge": {"kind": "spline", "variant": "base", "alpha": 100}, "svr_core": {"kind": "svr", "variant": "core", "C": 20}, "huber40_engineered": {"kind": "cat", "variant": "engineered", "depth": 5, "loss": "Huber:delta=40000000", "iterations": 619}, "huber40_monotone": {"kind": "cat", "variant": "base", "depth": 5, "loss": "Huber:delta=40000000", "monotone": true, "iterations": 447}}, "v2_weights": {"engineered": 0.06448907249408362, "engineered_drop": 0.09631834163153197, "ordered_depth5": 0.08507039177778712, "huber20_engineered": 0.06274452861282048, "huber40": 0.08855362607985372, "clip120": 0.07794631558814363, "spline_ridge": 0.15648185431084338, "svr_core": 0.058428294374292714, "huber40_engineered": 0.09327747966602483, "huber40_monotone": 0.2166900954646185}, "v3": {"v2_weight": 0.5, "additional_weights": {"bag_subsample_h40": 0.5}, "specs": {"bag_subsample_h40": {"variant": "base", "depth": 5, "iterations": 850, "delta": 40, "subsample": 0.7, "l2": 10}}, "model_seeds": [1042, 2042, 3042], "development": {"name": "bag_subsample_h40", "new_weight": 0.5, "rmse": 19839027.540841296, "gain": 10460.370932716876, "fold_wins": 3}, "baseline_development_rmse": 19849487.911774013, "development_split_seed": 42, "confirmation_split_seeds": [71, 2026, 303, 909], "selection": "Fixed configurations; seed42 development only. Coarse 25% blending grid. Seed909 unseen during this round selection, but all labeled rows were involved in previous development. No independent external test is claimed."}}')

V2_CONFIGS=CONTRACT['v2_configs']; V2_WEIGHTS=CONTRACT['v2_weights']; V3=CONTRACT['v3']
OLD_CONFIGS={
    'old_raw':dict(kind='cat',variant='base',depth=6,iterations=302,loss='RMSE'),
    'old_huber':dict(kind='cat',variant='base',depth=6,iterations=768,loss='Huber:delta=12000000')}
assert np.isclose(sum(V2_WEIGHTS.values()),1)
assert np.isclose(V3['v2_weight']+sum(V3['additional_weights'].values()),1)
variants={s['variant'] for s in V2_CONFIGS.values()}|{s['variant'] for s in V3['specs'].values()}
feature_data={v:features(train,v) for v in variants}
for v,X in feature_data.items():
    assert 'Annual Turnover' not in X and 'Registration Number' not in X
    assert X.columns.equals(features(test,v).columns)
    changed=train.copy();changed['Annual Turnover']=0;changed['Registration Number']=-1
    pd.testing.assert_frame_equal(X,features(changed,v))
assert np.isclose(rmse([0,3],[0,0]),np.sqrt(4.5))
print('PASS: original-scale RMSE toy example, weight sums, target/ID exclusion and feature alignment.')
print('V3 weights:',V3['v2_weight'],V3['additional_weights'])
print('Development-only comparison:',V3['development'])

## 5. Confirm fixed settings on repeated folds and unseen cities

Every row receives exactly one held-out prediction in each five-fold assignment.
The city stress test puts canonical city aliases in the same group, keeping city
groups disjoint between training and validation.

The local executed notebook can reuse the actual fold predictions from this round
using an explicit cache environment variable. Input hashes, configuration hash and
prediction-file hashes must match before reuse. The output identifies this reuse.
**Colab has no cache and executes the complete validation training below.**

In [ ]:
def get_splits(scheme):
    if scheme=='city':
        groups=features(train,'engineered')['City']
        splits=list(GroupKFold(5).split(train,groups=groups))
        for tr,va in splits: assert set(groups.iloc[tr]).isdisjoint(groups.iloc[va])
    else: splits=list(KFold(5,shuffle=True,random_state=int(scheme)).split(train))
    counts=np.zeros(len(y),int)
    for tr,va in splits:
        assert set(tr).isdisjoint(va); counts[va]+=1
    assert (counts==1).all()
    return splits

def compute_validation(scheme):
    splits=get_splits(scheme); results={k:np.zeros(len(y)) for k in ['old','v2','candidate']}
    for name,spec in {**OLD_CONFIGS,**V2_CONFIGS}.items():
        X=feature_data[spec['variant']]; component=np.zeros(len(y))
        for f,(tr,va) in enumerate(splits):
            component[va],_=fit_predict(X,y,tr,va,spec,43+f,spec.get('iterations'))
        if name in OLD_CONFIGS: results['old']+=(.55 if name=='old_raw' else .45)*component
        else: results['v2']+=V2_WEIGHTS[name]*component
    results['candidate']=V3['v2_weight']*results['v2']
    for name,w in V3['additional_weights'].items():
        spec=V3['specs'][name];X=feature_data[spec['variant']]; component=np.zeros(len(y))
        for f,(tr,va) in enumerate(splits):
            p=[]
            for seed in V3['model_seeds']:
                model=cat_fit(X.iloc[tr],y[tr],spec,seed+f)
                p.append(np.maximum(model.predict(X.iloc[va]),0))
            component[va]=np.mean(p,axis=0)
        results['candidate']+=w*component
    return results

cache_value=os.environ.get('MLA_VALIDATION_CACHE')
cache_root=Path(cache_value) if cache_value else None
manifest=None
if cache_root is not None:
    manifest=json.loads((cache_root/'cache_manifest.json').read_text())
    assert manifest['input_hashes']==input_hashes, 'Cache inputs changed'
    assert manifest['configuration_sha256']==hashlib.sha256(json.dumps(CONTRACT,sort_keys=True).encode()).hexdigest()
validation_predictions={};rows=[]
for scheme in [71,2026,303,909,'city']:
    get_splits(scheme)  # Check coverage and separation even when reusing predictions.
    filename=f'comparison_{scheme}.npz'
    if cache_root is not None:
        assert hashlib.sha256((cache_root/filename).read_bytes()).hexdigest()==manifest['files'][filename]
        cached=np.load(cache_root/filename)
        predictions={k:cached[k] for k in ['old','v2','candidate']}
        print('Verified executed-fold cache:',filename)
    else:
        print('Training validation models:',scheme)
        predictions=compute_validation(scheme)
    assert all(p.shape==y.shape and np.isfinite(p).all() and (p>=0).all() for p in predictions.values())
    validation_predictions[str(scheme)]=predictions
    rows.append({'Validation':str(scheme),'Submitted model RMSE':rmse(y,predictions['old']),
                 'Previous improved RMSE':rmse(y,predictions['v2']),'Current candidate RMSE':rmse(y,predictions['candidate'])})
validation_table=pd.DataFrame(rows)
validation_table['Gain vs previous improved']=validation_table['Previous improved RMSE']-validation_table['Current candidate RMSE']
print(validation_table.to_string(index=False,float_format=lambda x:f'{x:,.2f}'))
random_table=validation_table[validation_table['Validation']!='city']
means=random_table.drop(columns='Validation').mean()
print('Mean across four shuffled assignments:')
print(means.to_string(float_format=lambda x:f'{x:,.2f}'))

## 6. Interpret validation and uncertainty

Lower RMSE across several fixed split assignments supports trying the candidate.
The size of the improvement should be reported alongside uncertainty. The bootstrap
below resamples rows of already-computed errors; its interval is conditional on these
models and does not account for every source of model-selection or distribution risk.
The city result is a separate stress test, not part of the random-split average.
In this executed run the conditional interval includes zero, so the incremental
gain over v2 is not decisive under this resampling check.

In [ ]:
random_results=[validation_predictions[str(s)] for s in [71,2026,303,909]]
old_sq=np.mean([(y-p['old'])**2 for p in random_results],axis=0)
v2_sq=np.mean([(y-p['v2'])**2 for p in random_results],axis=0)
new_sq=np.mean([(y-p['candidate'])**2 for p in random_results],axis=0)
rng=np.random.default_rng(812);gains=[]
for _ in range(1000):
    ix=rng.integers(0,len(y),len(y));gains.append(np.sqrt(v2_sq[ix].mean())-np.sqrt(new_sq[ix].mean()))
gain_ci=np.quantile(gains,[.025,.975])
print('Conditional 95% interval for improvement vs v2:',np.round(gain_ci,2).tolist())
print('All targets included in the RMSE above; no hidden-test score is available.')
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
x=np.arange(len(validation_table)); labels=['71','2026','303','909','City']
for column,color in [('Submitted model RMSE','#64748b'),('Previous improved RMSE','#2563eb'),('Current candidate RMSE','#0f766e')]:
    axes[0].plot(x,validation_table[column]/1e6,'o-',label=column.replace(' RMSE',''),color=color,linewidth=1.5)
axes[0].set_xticks(x,labels)
axes[0].set(xlabel='Validation assignment',ylabel='RMSE (INR millions)',title='Absolute validation RMSE')
axes[0].legend(fontsize=8,loc='upper left');axes[0].grid(axis='y',alpha=.2)
g=validation_table['Gain vs previous improved'].to_numpy()/1000
axes[1].bar(x,g,color=['#0f766e' if v>=0 else '#b45309' for v in g])
axes[1].axhline(0,color='#334155',linewidth=1)
axes[1].set_xticks(x,labels)
axes[1].set(xlabel='Validation assignment',ylabel='RMSE reduction vs v2 (INR thousands)',title='Positive = improved; City stress test worsened')
fig.tight_layout();plt.show()

## 7. Fit the frozen models on all training rows

Training uses only the supplied train targets. The test data contributes no target
values. Predictors remain in original INR units; the final mixture is nonnegative.

In [ ]:
full_parts={};fitted_v2={};fitted_new={}
tr_idx=np.arange(len(train));te_idx=np.arange(len(train),len(train)+len(test))
combined_y=np.r_[y,np.full(len(test),np.nan)]
for name,spec in V2_CONFIGS.items():
    X=pd.concat([feature_data[spec['variant']],features(test,spec['variant'])],ignore_index=True)
    pred,trees,fitted=fit_predict(X,combined_y,tr_idx,te_idx,spec,42,spec.get('iterations'),return_fitted=True)
    full_parts[name]=pred;fitted_v2[name]=fitted
    print('Fitted v2 component:',name)
v2_test=sum(V2_WEIGHTS[n]*p for n,p in full_parts.items())
test_prediction=V3['v2_weight']*v2_test
for name,w in V3['additional_weights'].items():
    spec=V3['specs'][name];models=[];parts=[]
    for seed in V3['model_seeds']:
        model=cat_fit(feature_data[spec['variant']],y,spec,seed)
        models.append(model);parts.append(np.maximum(model.predict(features(test,spec['variant'])),0))
    fitted_new[name]=models;test_prediction+=w*np.mean(parts,axis=0)
    print('Fitted new component:',name,'seeds:',V3['model_seeds'])

def predict_final(raw):
    prev=sum(V2_WEIGHTS[n]*predict_fitted(m,features(raw,V2_CONFIGS[n]['variant'])) for n,m in fitted_v2.items())
    pred=V3['v2_weight']*prev
    for name,w in V3['additional_weights'].items():
        X=features(raw,V3['specs'][name]['variant'])
        pred+=w*np.mean([np.maximum(m.predict(X),0) for m in fitted_new[name]],axis=0)
    return pred

assert np.allclose(predict_final(test),test_prediction,rtol=1e-12,atol=.01)
print('Mean absolute change from v2 predictions (INR):',float(np.abs(test_prediction-v2_test).mean()))

## 8. Behavior tests, exact submission format, and export

The CSV contains 500 prediction records plus one header row. No index column is
written. These checks establish format and prediction behavior; they do not measure
accuracy on the test restaurants because their true turnovers are unavailable.

In [ ]:
perm=np.random.default_rng(91).permutation(len(test))
assert np.allclose(predict_final(test.iloc[perm]),test_prediction[perm],rtol=1e-10,atol=.01)
edge=test.iloc[:2].copy();edge['City']='unseen_validation_city';edge['Cuisine']=pd.NA
edge['Facebook Popularity Quotient']=np.nan;edge['Live Music Rating']=np.nan
edge_pred=predict_final(edge)
assert np.isfinite(edge_pred).all() and (edge_pred>=0).all()

def check_submission(frame):
    assert frame.shape==(500,2)
    assert frame.columns.tolist()==['Registration Number','Annual Turnover']
    assert frame['Registration Number'].equals(test['Registration Number'])
    assert frame['Registration Number'].is_unique
    assert np.isfinite(frame['Annual Turnover']).all() and (frame['Annual Turnover']>=0).all()

submission=pd.DataFrame({'Registration Number':test['Registration Number'],'Annual Turnover':test_prediction})
check_submission(submission)
# Deliberately wrong row order, count, header and values must be rejected.
bad_cases=[submission.iloc[:-1],submission.iloc[::-1].reset_index(drop=True),
           submission.rename(columns={'Annual Turnover':'Turnover'}),submission.assign(index=0)]
for invalid in [np.nan,np.inf,-1]:
    b=submission.copy();b.loc[0,'Annual Turnover']=invalid;bad_cases.append(b)
for bad in bad_cases:
    rejected=False
    try:check_submission(bad)
    except AssertionError:rejected=True
    assert rejected,'Invalid submission was not rejected'
output_path=OUTPUT_DIR/(FILE_STEM+'.csv')
submission.to_csv(output_path,index=False)
roundtrip=pd.read_csv(output_path);check_submission(roundtrip)
assert np.allclose(roundtrip['Annual Turnover'],test_prediction,rtol=1e-12,atol=.001)
assert len(output_path.read_text().splitlines())==501
print('PASS: row permutation, unseen/missing values, seven intentionally invalid examples, exact headers, IDs,500 rows, finite values and CSV round-trip.')
print('Saved:',output_path.name)
print('SHA256:',hashlib.sha256(output_path.read_bytes()).hexdigest())
print('Turnover range:',float(test_prediction.min()),'to',float(test_prediction.max()))
print(submission.head().to_string(index=False))

## 9. Inferences and submission

- The previous submitted model and improved candidates are compared on identical
  validation rows, using the same original-scale RMSE formula.
- Robust training and blending reduce sensitivity to extreme targets and training
  randomness. The resulting gains are modest; adding algorithms did not always help.
- The current candidate combines the existing mixture with three subsampled models.
  Its apparent local improvement is evidence for a submission, not proof of rank 1.
- The city-isolation stress test worsened relative to v2. This is a real limitation:
  generalization to entirely unseen cities may be weaker even though all four random
  splits improved. Both improved candidates still beat the original model there.
- The conditional bootstrap interval includes zero. Treat the small gain over v2
  as uncertain and this file as a candidate to evaluate on the portal.
- A new portal score below **11,946,659.0702364** would beat the leader in the supplied
  screenshot, assuming that leader score remains unchanged. The new result is unknown.

Upload the generated **CSV** under **assignment M.Tech. MLA Hackathon -Restaurant Turnover
Prediction**. Keep the observed portal score with that exact file version.
Upload the executed **notebook** with its code, outputs and inferences to
**LLJ_Oct_2026** only when the model choice is final; that assignment permits one
submission according to the provided instructions. The hackathon permits repeated
CSV submissions. The portal registration format is a separate faculty requirement;
renaming these files does not change the registration shown as submission.

In [ ]:
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print('Prediction CSV is ready in the notebook folder.')